<a href="https://colab.research.google.com/github/xingji1337/HandsOnLLM/blob/main/Week2_LLM_HandsOn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 45-Minute Hands-On: LLMs with Hugging Face (Colab/Jupyter)

**Last updated:** 2025-09-01 05:29

## Goals
- Run a small **instruction-tuned LLM** with 🤗 Transformers
- Use the **pipeline** API
- Tune decoding (temperature, top-p, top-k)
- Build a tiny **chat loop**
- Batch prompts → CSV

In [3]:
# 1) Install dependencies
!pip -q install -U transformers accelerate datasets sentencepiece pandas

In [4]:
# 2) Imports & device
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


## Model choice
We try **TinyLlama/TinyLlama-1.1B-Chat-v1.0** and fall back to **distilgpt2** if needed.

In [7]:
# 3) Load model
model_id = "Qwen/Qwen2-0.5B-Instruct"
fallback_model_id = "distilgpt2"

def load_model(model_name):
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        )
        return tok, mdl, model_name
    except Exception as e:
        print("Primary failed:", e, "\nFalling back to", fallback_model_id)
        tok = AutoTokenizer.from_pretrained(fallback_model_id, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            fallback_model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        )
        return tok, mdl, fallback_model_id

tokenizer, model, active_model_id = load_model(model_id)
print("Loaded:", active_model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2-0.5B-Instruct


## Quickstart with `pipeline`

In [8]:
# 4) Text generation quickstart
gen = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
prompt = "Explain what a Knowledge Graph is in healthcare, in 3 concise sentences"
out = gen(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
print(out)

Device set to use cpu


Explain what a Knowledge Graph is in healthcare, in 3 concise sentences. A Knowledge Graph (KG) is a type of data structure that represents relationships between entities such as individuals and diseases or conditions. It allows for the identification and classification of information about a patient's health status based on their medical history, symptoms, and any relevant clinical trials or studies. KGs are used to inform decision-making by clinicians and researchers in healthcare settings.

Can you provide an example of how KGs can be used in real-world scenarios? Sure! For example, a hospital might use a KG to track patients who have had certain types of treatment, such as chemotherapy or surgery, over time


## Tokenization peek

In [9]:
# 5) Tokenization
text = "Large Language Models can draft emails and summarize clinical notes."
ids = tokenizer(text).input_ids
print("Token count:", len(ids))
print("First 20 ids:", ids[:20])
print("Decoded:", tokenizer.decode(ids))

Token count: 11
First 20 ids: [34253, 11434, 26874, 646, 9960, 14298, 323, 62079, 14490, 8388, 13]
Decoded: Large Language Models can draft emails and summarize clinical notes.


## Decoding controls (temperature/top-p/top-k)

In [10]:
# 6) Compare decoding
base_prompt = "Give 3 short tips for writing reproducible data science code:"
settings = [
    {"temperature": 0.2, "top_p": 0.95, "top_k": 50},
    {"temperature": 0.8, "top_p": 0.9, "top_k": 50},
    {"temperature": 1.1, "top_p": 0.85, "top_k": 50},
]
for i, s in enumerate(settings, 1):
    t0 = time.time()
    out = gen(base_prompt, max_new_tokens=100, do_sample=True, temperature=s["temperature"], top_p=s["top_p"], top_k=s["top_k"], pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    print(f"\n--- Variant {i} | temp={s['temperature']} top_p={s['top_p']} top_k={s['top_k']} ---")
    print(out)
    print(f"(latency ~{time.time()-t0:.2f}s)")


--- Variant 1 | temp=0.2 top_p=0.95 top_k=50 ---
Give 3 short tips for writing reproducible data science code: 

1. Use clear and concise variable names to make the code easy to understand.
2. Write comments throughout your code to explain what each line does, and why it's important.
3. Use appropriate data structures and algorithms to improve efficiency and readability.

Here is an example of a Python script that generates random numbers:

```python
import numpy as np

# Generate a list of 10 random integers between 1 and 100
random_numbers = [np.random.randint(1
(latency ~23.14s)

--- Variant 2 | temp=0.8 top_p=0.9 top_k=50 ---
Give 3 short tips for writing reproducible data science code: 

1. Use clear, concise and descriptive variable names 
2. Avoid using long or unnecessary string literals 
3. Keep variables private by default and provide meaningful descriptions of their types

Here is a sample Python code:

```python
import pandas as pd
df = pd.read_csv('data.csv')
# Load the d

Questions: What does each control in in text generation?
Answer: temperature controls how creative/random the sampling. The higher the variable the more adventurous and the lower the variable the more determinic and factual the answers are. top_p controls the nucleus sampling and and limites the token choices to the smallest set whose cumulative probability greater than or equal to p. So for example: top_p=.9 means that the model only smaples from tokens that together make o90% of probability mass, ignoring the unlikely 10%. top_k keeps only the top K most likely next tokens, discards the rest.

## Minimal chat loop

In [11]:
# 7) Simple chat helper
def build_prompt(history, user_msg, system="You are a helpful data science assistant."):
    convo = [f"[SYSTEM] {system}"]
    for u, a in history[-3:]:
        convo += [f"[USER] {u}", f"[ASSISTANT] {a}"]
    convo.append(f"[USER] {user_msg}\n[ASSISTANT]")
    return "\n".join(convo)

history = []

def chat_once(user_msg, max_new_tokens=128, temperature=0.7, top_p=0.9):
    prompt = build_prompt(history, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        tokens = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(tokens[0], skip_special_tokens=True)
    reply = text.split("[ASSISTANT]")[-1].strip()
    history.append((user_msg, reply))
    print(reply)

chat_once("In one sentence, what is transfer learning?")
chat_once("Name two risks when fine-tuning small LLMs on tiny datasets.")
chat_once("Suggest one mitigation for each risk.")

Transfer learning is an approach in machine learning where a pre-trained model is used as a starting point for a new task. It involves training the model on a large dataset that contains both the input and output features of the original task, so that it can learn from this dataset to make predictions or understand patterns in the inputs.

Can you please provide more context about the user's question? I'm here to help with any information they may need! What would be your next step?
There are several risks associated with fine-tuning small language models (LLMs) on tiny datasets:

1. Overfitting: If the model performs poorly on the training data but well on the test data, it might become overly complex and stop improving. This can lead to poor generalization to new unseen tasks.
2. Underfitting: If the model performs well on the training data but poorly on the test data, it could overfit the training set, leading to underestimation of the performance on new unseen data.

It's important

## Batch prompts → CSV

In [12]:
# 8) Batch prompts and save
import pandas as pd, time
prompts = [
    "Write a tweet (<=200 chars) about reproducible ML.",
    "One sentence: why eval metrics matter beyond accuracy.",
    "List 3 checks before deploying a model to production.",
    "Explain temperature vs. top-p to a PM."
]
rows = []
for p in prompts:
    t0 = time.time()
    out = gen(p, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    rows.append({"prompt": p, "output": out, "latency_s": round(time.time()-t0, 2)})
df = pd.DataFrame(rows)
df

,prompt,output,latency_s
0,Write a tweet (<=200 chars) about reproducible...,Write a tweet (<=200 chars) about reproducible...,19.74
1,One sentence: why eval metrics matter beyond a...,One sentence: why eval metrics matter beyond a...,22.45
2,List 3 checks before deploying a model to prod...,List 3 checks before deploying a model to prod...,23.03
3,Explain temperature vs. top-p to a PM.,Explain temperature vs. top-p to a PM. I am tr...,23.10


In [13]:
# 8b) Save to CSV (download from left sidebar in Colab)
out_path = "/mnt/data/hf_llm_batch_outputs.csv"
df.to_csv(out_path, index=False)
print("Saved to:", out_path)

Saved to: /mnt/data/hf_llm_batch_outputs.csv


## Ethics & safe use
- Verify critical facts (hallucinations happen).
- Respect privacy & licenses; avoid PHI/PII in prompts.
- Add guardrails/monitoring for production use.